# 🧠 Notebook 2: Model Training
## Emotion-Conditioned Image Captioning

---

## ✅ What This Notebook Does
Trains **4 model variants** for the ablation study:
| # | Variant | Description |
|---|---------|-------------|
| 1 | **Baseline** | No emotion conditioning — plain LSTM caption model |
| 2 | **Emotion@Start** | Emotion fused into LSTM initial hidden state only |
| 3 | **Emotion@Every** | Emotion concatenated with word embedding at every step |
| 4 | **CrossAttention** | Emotion injected via cross-attention at every step |

## ⚠️ Before You Run
- Make sure **Runtime → Change runtime type → T4 GPU** is selected
- **Run Notebook 1 first!** This notebook reads data saved by Notebook 1
- Each variant trains for ~15–20 min on free Colab
- Training all 4 = ~60–80 min total — you can train one at a time if Colab disconnects
- Enable **Colab Notifications** (bell icon) to know when cells finish

---

## Step 1 — Mount Drive & Load Config

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, pickle, sys
PROJECT_DIR = '/content/drive/MyDrive/EmotionCaptioning'

with open(f'{PROJECT_DIR}/data/config.pkl', 'rb') as f:
    config = pickle.load(f)
with open(f'{PROJECT_DIR}/data/vocab.pkl', 'rb') as f:
    vocab_data = pickle.load(f)

vocab       = vocab_data['vocab']
idx_to_word = vocab_data['idx_to_word']
EMOTIONS    = vocab_data['emotions']
IMG_DIR     = config['IMG_DIR']

print('✅ Config loaded')
print('Vocab size:', config['VOCAB_SIZE'])
print('Emotions:', EMOTIONS)

## Step 2 — Install Dependencies

In [ ]:
!pip install -q nltk
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
print('✅ Done')

## Step 3 — Dataset & DataLoader

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
import pandas as pd
import numpy as np
from PIL import Image
import re

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

VOCAB_SIZE  = config['VOCAB_SIZE']
N_EMOTIONS  = config['N_EMOTIONS']
EMBED_DIM   = config['EMBED_DIM']
HIDDEN_DIM  = config['HIDDEN_DIM']
VISUAL_DIM  = config['VISUAL_DIM']
FUSED_DIM   = config['FUSED_DIM']
MAX_SEQ_LEN = config['MAX_SEQ_LEN']
PAD_IDX     = vocab['<PAD>']
SOS_IDX     = vocab['<SOS>']
EOS_IDX     = vocab['<EOS>']
UNK_IDX     = vocab['<UNK>']

In [ ]:
def tokenize(text):
    text = text.lower().strip()
    text = re.sub(r"[^a-z0-9\s']", '', text)
    return text.split()

def caption_to_ids(caption, vocab, max_len=MAX_SEQ_LEN):
    tokens = tokenize(caption)
    ids = [SOS_IDX] + [vocab.get(t, UNK_IDX) for t in tokens] + [EOS_IDX]
    ids = ids[:max_len]
    ids += [PAD_IDX] * (max_len - len(ids))
    return ids

IMG_TRANSFORMS = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

class EmotionCaptionDataset(Dataset):
    def __init__(self, csv_path, vocab, img_dir, transform=IMG_TRANSFORMS):
        self.df        = pd.read_csv(csv_path)
        self.vocab     = vocab
        self.img_dir   = img_dir
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        img_path = row['img_path'] if os.path.isabs(row['img_path']) else os.path.join(self.img_dir, row['image_id'])
        img      = Image.open(img_path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        caption  = caption_to_ids(row['emotion_caption'], self.vocab)
        emotion  = int(row['emotion_id'])
        return img, torch.tensor(caption, dtype=torch.long), torch.tensor(emotion, dtype=torch.long)

# Load datasets
BATCH_SIZE = 32  # Good for T4 16GB; reduce to 16 if OOM

train_ds = EmotionCaptionDataset(f'{PROJECT_DIR}/data/train.csv', vocab, IMG_DIR)
val_ds   = EmotionCaptionDataset(f'{PROJECT_DIR}/data/val.csv',   vocab, IMG_DIR)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'✅ Train: {len(train_ds)} | Val: {len(val_ds)}')
print(f'   Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')

## Step 4 — Visual Encoder (Frozen ResNet-50)

We freeze ALL ResNet weights. This saves GPU memory and matches the BLIP-2 paradigm.

In [ ]:
class FrozenVisualEncoder(nn.Module):
    """ResNet-50 with the final FC layer removed. Returns 2048-d feature vector."""
    def __init__(self):
        super().__init__()
        resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        # Remove final classification layer
        self.encoder = nn.Sequential(*list(resnet.children())[:-1])
        # Freeze ALL weights
        for param in self.encoder.parameters():
            param.requires_grad = False

    def forward(self, x):
        with torch.no_grad():
            feat = self.encoder(x)          # (B, 2048, 1, 1)
        return feat.view(feat.size(0), -1)  # (B, 2048)

visual_encoder = FrozenVisualEncoder().to(device)
trainable = sum(p.numel() for p in visual_encoder.parameters() if p.requires_grad)
total     = sum(p.numel() for p in visual_encoder.parameters())
print(f'✅ Visual encoder: {total:,} params total, {trainable:,} trainable (all frozen)')

## Step 5 — Model Architectures (All 4 Variants)

All variants share the same visual encoder. What changes is how and when the emotion is injected.

In [ ]:
# ============================================================
# VARIANT 1: Baseline — No emotion conditioning
# ============================================================
class BaselineCaptioner(nn.Module):
    def __init__(self, vocab_size, embed_dim=256, hidden_dim=512, visual_dim=2048):
        super().__init__()
        self.projection = nn.Sequential(
            nn.Linear(visual_dim, hidden_dim),
            nn.ReLU()
        )
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.lstm      = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc_out    = nn.Linear(hidden_dim, vocab_size)

    def forward(self, visual_feat, captions, _emotion_ids=None):
        h0 = self.projection(visual_feat).unsqueeze(0)  # (1, B, H)
        c0 = torch.zeros_like(h0)
        embeds, _  = self.lstm(self.embedding(captions[:, :-1]), (h0, c0))
        return self.fc_out(embeds)  # (B, T-1, V)

    def generate(self, visual_feat, emotion_id=None, max_len=MAX_SEQ_LEN):
        B = visual_feat.size(0)
        h = self.projection(visual_feat).unsqueeze(0)
        c = torch.zeros_like(h)
        token = torch.full((B, 1), SOS_IDX, dtype=torch.long, device=device)
        results = []
        for _ in range(max_len):
            emb         = self.embedding(token)
            out, (h, c) = self.lstm(emb, (h, c))
            logits      = self.fc_out(out.squeeze(1))
            token       = logits.argmax(dim=-1, keepdim=True)
            results.append(token)
            if (token == EOS_IDX).all(): break
        return torch.cat(results, dim=1)


# ============================================================
# VARIANT 2: Emotion@Start — emotion fused into hidden state
# ============================================================
class EmotionAtStartCaptioner(nn.Module):
    def __init__(self, vocab_size, n_emotions=5, embed_dim=256, emotion_dim=64, hidden_dim=512, visual_dim=2048):
        super().__init__()
        self.emotion_embed = nn.Embedding(n_emotions, emotion_dim)
        self.projection    = nn.Sequential(
            nn.Linear(visual_dim + emotion_dim, hidden_dim),
            nn.ReLU()
        )
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.lstm      = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc_out    = nn.Linear(hidden_dim, vocab_size)

    def forward(self, visual_feat, captions, emotion_ids):
        emo   = self.emotion_embed(emotion_ids)          # (B, E)
        fused = torch.cat([visual_feat, emo], dim=-1)   # (B, V+E)
        h0    = self.projection(fused).unsqueeze(0)     # (1, B, H)
        c0    = torch.zeros_like(h0)
        embeds, _ = self.lstm(self.embedding(captions[:, :-1]), (h0, c0))
        return self.fc_out(embeds)

    def generate(self, visual_feat, emotion_id, max_len=MAX_SEQ_LEN):
        B   = visual_feat.size(0)
        emo = self.emotion_embed(emotion_id)
        fused = torch.cat([visual_feat, emo], dim=-1)
        h   = self.projection(fused).unsqueeze(0)
        c   = torch.zeros_like(h)
        token = torch.full((B, 1), SOS_IDX, dtype=torch.long, device=device)
        results = []
        for _ in range(max_len):
            emb         = self.embedding(token)
            out, (h, c) = self.lstm(emb, (h, c))
            logits      = self.fc_out(out.squeeze(1))
            token       = logits.argmax(dim=-1, keepdim=True)
            results.append(token)
            if (token == EOS_IDX).all(): break
        return torch.cat(results, dim=1)


# ============================================================
# VARIANT 3: Emotion@Every — persistent conditioning
# ============================================================
class EmotionAtEveryCaptioner(nn.Module):
    def __init__(self, vocab_size, n_emotions=5, word_embed_dim=256, emotion_dim=64, hidden_dim=512, visual_dim=2048):
        super().__init__()
        self.emotion_embed = nn.Embedding(n_emotions, emotion_dim)
        self.vis_proj      = nn.Sequential(
            nn.Linear(visual_dim + emotion_dim, hidden_dim),
            nn.ReLU()
        )
        self.word_embed = nn.Embedding(vocab_size, word_embed_dim, padding_idx=PAD_IDX)
        # LSTM input: word_embed + emotion_embed concatenated
        self.lstm       = nn.LSTM(word_embed_dim + emotion_dim, hidden_dim, batch_first=True)
        self.fc_out     = nn.Linear(hidden_dim, vocab_size)

    def forward(self, visual_feat, captions, emotion_ids):
        B, T = captions.shape[0], captions.shape[1] - 1
        emo   = self.emotion_embed(emotion_ids)                        # (B, E)
        fused = torch.cat([visual_feat, emo], dim=-1)
        h0    = self.vis_proj(fused).unsqueeze(0)
        c0    = torch.zeros_like(h0)
        word_embs = self.word_embed(captions[:, :-1])                  # (B, T, W)
        emo_rep   = emo.unsqueeze(1).expand(-1, T, -1)                 # (B, T, E)
        lstm_in   = torch.cat([word_embs, emo_rep], dim=-1)           # (B, T, W+E)
        out, _    = self.lstm(lstm_in, (h0, c0))
        return self.fc_out(out)

    def generate(self, visual_feat, emotion_id, max_len=MAX_SEQ_LEN):
        B   = visual_feat.size(0)
        emo = self.emotion_embed(emotion_id)
        h   = self.vis_proj(torch.cat([visual_feat, emo], dim=-1)).unsqueeze(0)
        c   = torch.zeros_like(h)
        token = torch.full((B, 1), SOS_IDX, dtype=torch.long, device=device)
        results = []
        for _ in range(max_len):
            word_emb    = self.word_embed(token)
            lstm_in     = torch.cat([word_emb, emo.unsqueeze(1)], dim=-1)
            out, (h, c) = self.lstm(lstm_in, (h, c))
            token       = self.fc_out(out.squeeze(1)).argmax(dim=-1, keepdim=True)
            results.append(token)
            if (token == EOS_IDX).all(): break
        return torch.cat(results, dim=1)


# ============================================================
# VARIANT 4: CrossAttention fusion
# ============================================================
class CrossAttentionFusion(nn.Module):
    """Allows the LSTM hidden state to attend to the emotion embedding at every step."""
    def __init__(self, hidden_dim=512, emotion_dim=64):
        super().__init__()
        self.attn = nn.MultiheadAttention(embed_dim=hidden_dim, num_heads=4,
                                           kdim=emotion_dim, vdim=emotion_dim,
                                           batch_first=True)
        self.norm = nn.LayerNorm(hidden_dim)

    def forward(self, query, key_value):
        # query: (B, 1, H), key_value: (B, 1, E)
        attn_out, _ = self.attn(query, key_value, key_value)
        return self.norm(query + attn_out)

class CrossAttentionCaptioner(nn.Module):
    def __init__(self, vocab_size, n_emotions=5, word_embed_dim=256, emotion_dim=64, hidden_dim=512, visual_dim=2048):
        super().__init__()
        self.emotion_embed = nn.Embedding(n_emotions, emotion_dim)
        self.vis_proj      = nn.Sequential(
            nn.Linear(visual_dim, hidden_dim),
            nn.ReLU()
        )
        self.word_embed  = nn.Embedding(vocab_size, word_embed_dim, padding_idx=PAD_IDX)
        self.lstm        = nn.LSTMCell(word_embed_dim, hidden_dim)
        self.cross_attn  = CrossAttentionFusion(hidden_dim, emotion_dim)
        self.fc_out      = nn.Linear(hidden_dim, vocab_size)

    def forward(self, visual_feat, captions, emotion_ids):
        B, T = captions.shape[0], captions.shape[1] - 1
        h    = self.vis_proj(visual_feat)          # (B, H)
        c    = torch.zeros_like(h)
        emo  = self.emotion_embed(emotion_ids).unsqueeze(1)  # (B, 1, E)
        outputs = []
        for t in range(T):
            word_emb    = self.word_embed(captions[:, t])    # (B, W)
            h, c        = self.lstm(word_emb, (h, c))
            h_attended  = self.cross_attn(h.unsqueeze(1), emo).squeeze(1)
            outputs.append(self.fc_out(h_attended))
        return torch.stack(outputs, dim=1)  # (B, T, V)

    def generate(self, visual_feat, emotion_id, max_len=MAX_SEQ_LEN):
        B    = visual_feat.size(0)
        h    = self.vis_proj(visual_feat)
        c    = torch.zeros_like(h)
        emo  = self.emotion_embed(emotion_id).unsqueeze(1)
        token = torch.full((B,), SOS_IDX, dtype=torch.long, device=device)
        results = []
        for _ in range(max_len):
            word_emb   = self.word_embed(token)
            h, c       = self.lstm(word_emb, (h, c))
            h_att      = self.cross_attn(h.unsqueeze(1), emo).squeeze(1)
            logits     = self.fc_out(h_att)
            token      = logits.argmax(dim=-1)
            results.append(token.unsqueeze(1))
            if (token == EOS_IDX).all(): break
        return torch.cat(results, dim=1)

print('✅ All 4 model architectures defined')

## Step 6 — Training Loop

In [ ]:
import time
from tqdm.auto import tqdm

def train_one_epoch(model, encoder, loader, optimizer, criterion):
    model.train()
    total_loss, n = 0, 0
    for imgs, captions, emotions in loader:
        imgs, captions, emotions = imgs.to(device), captions.to(device), emotions.to(device)
        visual_feat = encoder(imgs)
        logits      = model(visual_feat, captions, emotions)  # (B, T-1, V)
        targets     = captions[:, 1:].reshape(-1)              # (B*(T-1),)
        loss        = criterion(logits.reshape(-1, VOCAB_SIZE), targets)
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        n           += imgs.size(0)
    return total_loss / n

@torch.no_grad()
def evaluate(model, encoder, loader, criterion):
    model.eval()
    total_loss, n = 0, 0
    for imgs, captions, emotions in loader:
        imgs, captions, emotions = imgs.to(device), captions.to(device), emotions.to(device)
        visual_feat = encoder(imgs)
        logits      = model(visual_feat, captions, emotions)
        targets     = captions[:, 1:].reshape(-1)
        loss        = criterion(logits.reshape(-1, VOCAB_SIZE), targets)
        total_loss += loss.item() * imgs.size(0)
        n           += imgs.size(0)
    return total_loss / n

def train_model(model, name, n_epochs=15, lr=3e-4):
    criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)

    best_val_loss = float('inf')
    ckpt_path     = f'{PROJECT_DIR}/checkpoints/{name}_best.pt'
    history       = {'train_loss': [], 'val_loss': []}

    print(f'\n======== Training: {name} ========')
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'Trainable parameters: {trainable_params:,}')

    for epoch in range(1, n_epochs + 1):
        t0       = time.time()
        tr_loss  = train_one_epoch(model, visual_encoder, train_loader, optimizer, criterion)
        val_loss = evaluate(model, visual_encoder, val_loader, criterion)
        scheduler.step(val_loss)
        history['train_loss'].append(tr_loss)
        history['val_loss'].append(val_loss)
        elapsed  = time.time() - t0
        print(f'Epoch {epoch:2d}/{n_epochs}  train_loss={tr_loss:.4f}  val_loss={val_loss:.4f}  [{elapsed:.0f}s]')
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), ckpt_path)
            print(f'           ↑ Best model saved to {ckpt_path}')

    model.load_state_dict(torch.load(ckpt_path))
    return model, history

print('✅ Training functions defined')

## Step 7 — Train Variant 1: Baseline (No Emotion)

In [ ]:
model_baseline = BaselineCaptioner(VOCAB_SIZE, embed_dim=256, hidden_dim=HIDDEN_DIM, visual_dim=VISUAL_DIM).to(device)
model_baseline, hist_baseline = train_model(model_baseline, 'baseline', n_epochs=15)

## Step 8 — Train Variant 2: Emotion@Start

In [ ]:
model_start = EmotionAtStartCaptioner(VOCAB_SIZE, N_EMOTIONS, embed_dim=256, emotion_dim=EMBED_DIM,
                                       hidden_dim=HIDDEN_DIM, visual_dim=VISUAL_DIM).to(device)
model_start, hist_start = train_model(model_start, 'emotion_at_start', n_epochs=15)

## Step 9 — Train Variant 3: Emotion@Every (Persistent Conditioning)

In [ ]:
model_every = EmotionAtEveryCaptioner(VOCAB_SIZE, N_EMOTIONS, word_embed_dim=256, emotion_dim=EMBED_DIM,
                                       hidden_dim=HIDDEN_DIM, visual_dim=VISUAL_DIM).to(device)
model_every, hist_every = train_model(model_every, 'emotion_at_every', n_epochs=15)

## Step 10 — Train Variant 4: Cross-Attention Fusion

In [ ]:
model_xattn = CrossAttentionCaptioner(VOCAB_SIZE, N_EMOTIONS, word_embed_dim=256, emotion_dim=EMBED_DIM,
                                       hidden_dim=HIDDEN_DIM, visual_dim=VISUAL_DIM).to(device)
model_xattn, hist_xattn = train_model(model_xattn, 'cross_attention', n_epochs=15)

## Step 11 — Plot Training Curves

In [ ]:
import matplotlib.pyplot as plt

histories = {
    'Baseline':        hist_baseline,
    'Emotion@Start':   hist_start,
    'Emotion@Every':   hist_every,
    'CrossAttention':  hist_xattn,
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for name, h in histories.items():
    axes[0].plot(h['train_loss'], label=name)
    axes[1].plot(h['val_loss'],   label=name)

axes[0].set_title('Training Loss');   axes[0].set_xlabel('Epoch'); axes[0].legend()
axes[1].set_title('Validation Loss'); axes[1].set_xlabel('Epoch'); axes[1].legend()
plt.tight_layout()
plt.savefig(f'{PROJECT_DIR}/results/training_curves.png', dpi=150)
plt.show()
print('✅ Training curves saved')

## Step 12 — Save Model Architectures for Notebook 3

In [ ]:
import pickle

# Save histories
with open(f'{PROJECT_DIR}/results/training_histories.pkl', 'wb') as f:
    pickle.dump(histories, f)

print('✅ All checkpoints and histories saved to Google Drive')
print('   Checkpoints:')
for fname in os.listdir(f'{PROJECT_DIR}/checkpoints'):
    fpath = f'{PROJECT_DIR}/checkpoints/{fname}'
    print(f'   {fname}  ({os.path.getsize(fpath)/1e6:.1f} MB)')

## Step 13 — Quick Inference Demo

In [ ]:
import pandas as pd
from PIL import Image

def ids_to_caption(token_ids, idx_to_word):
    words = []
    for idx in token_ids:
        w = idx_to_word.get(int(idx), '<UNK>')
        if w in ('<EOS>', '<PAD>'): break
        if w not in ('<SOS>',):
            words.append(w)
    return ' '.join(words)

# Pick a test image
df_test = pd.read_csv(f'{PROJECT_DIR}/data/test.csv')
row     = df_test.iloc[0]
img_pil = Image.open(row['img_path']).convert('RGB')
img_t   = IMG_TRANSFORMS(img_pil).unsqueeze(0).to(device)

visual_encoder.eval()
with torch.no_grad():
    feat = visual_encoder(img_t)

print('Test image:', row['image_id'])
print('Neutral caption:', row['neutral_caption'])
print()

for emotion_name, emotion_idx in config['EMOTION_TO_ID'].items():
    eid = torch.tensor([emotion_idx], device=device)
    for model_name, model in [('Emotion@Every', model_every), ('CrossAttn', model_xattn)]:
        tokens = model.generate(feat, eid)
        cap    = ids_to_caption(tokens[0].cpu().tolist(), idx_to_word)
        print(f'{model_name:15s} | {emotion_name:10s}: {cap}')

---
## ✅ Notebook 2 Complete!

### What was saved:
| File | Contents |
|------|----------|
| `checkpoints/baseline_best.pt`       | Baseline model weights |
| `checkpoints/emotion_at_start_best.pt` | Variant 2 weights |
| `checkpoints/emotion_at_every_best.pt` | Variant 3 weights |
| `checkpoints/cross_attention_best.pt`  | Variant 4 weights |
| `results/training_curves.png`        | Loss curves |
| `results/training_histories.pkl`     | Loss history data |

### ➡️ Next Step
Open **Notebook 3: Evaluation & Analysis** to compute BLEU/CIDEr, visualize attention, and do emotion interpolation.